## Lógica Guía 3

Resumen de las lógicas y herramientas SQL usadas para resolver los ejercicios de `GUIA3.ipynb`.

La idea de este archivo no es repetir cada respuesta, sino explicar **qué patrón lógico se usó**, **por qué sirve** y **cómo reconocer cuándo conviene usarlo**.

### Herramientas SQL usadas

- `SELECT`: elegir qué columnas se quieren mostrar.
- `FROM`: indicar de qué tabla salen los datos principales.
- `WHERE`: filtrar filas según una condición.
- `ORDER BY`: ordenar el resultado.
- `LIMIT`: quedarse solo con cierta cantidad de filas.
- `DISTINCT`: evitar repetidos.
- `JOIN`: relacionar tablas cuando la información está repartida.
- `SUM`, `COUNT`, `AVG`, `MIN`, `MAX`: hacer cálculos sobre varias filas.
- `GROUP BY`: agrupar antes de calcular.
- `HAVING`: filtrar después de agrupar.
- Subconsultas: usar el resultado de una consulta dentro de otra.
- `IN` / `NOT IN`: comparar contra un conjunto de valores.
- `EXISTS` / `NOT EXISTS`: verificar si existe o no existe una relación.
- `ROUND`: redondear resultados numéricos.

Regla práctica:

- si la consigna nombra una sola tabla, suele alcanzar con `SELECT` + `WHERE` + `ORDER BY`;
- si mezcla entidades distintas, suele aparecer `JOIN`;
- si pide totales, promedios, máximos o mínimos, hay agregación;
- si habla de "al menos uno", "ninguno" o "todos", casi siempre aparece una subconsulta o una lógica de existencia.

### Lógicas base

- **Proyección**: mostrar solo las columnas que pide la consigna.
- **Filtro simple**: quedarse con las filas que cumplan una condición directa.
- **Ordenamiento**: decidir si importa el valor más grande, el más chico o el orden alfabético.
- **Corte de resultados**: combinar `ORDER BY` con `LIMIT` para top 5, top 7, peor 5, etc.
- **Relación entre tablas**: usar `JOIN` cuando la información no vive en una sola tabla.
- **Eliminación de repetidos**: usar `DISTINCT` si una relación hace que la misma entidad aparezca varias veces.
- **Agregación**: resumir muchas filas en un total, un promedio, un mínimo o un máximo.
- **Existencia**: expresar que una entidad tiene al menos una relación que cumple algo.
- **Universalidad**: expresar que todas las relaciones deben cumplir algo.
- **Exclusión**: encontrar lo que no tiene relación o lo que no cumple cierta condición.

Ejemplo simple:

```sql
SELECT nombre, pais_origen
FROM bandas
WHERE genero = 'Rock';
```

Acá la lógica es:

- proyectar solo `nombre` y `pais_origen`;
- filtrar con `WHERE`;
- no hace falta `JOIN` porque todo sale de la misma tabla.

### Subconsultas: qué son y para qué sirven

Una **subconsulta** es una consulta metida dentro de otra. Se usa cuando una parte del problema depende del resultado de otra consulta intermedia.

Sirven mucho cuando la consigna pide cosas como:

- las bandas que **tienen al menos un** álbum con cierta condición;
- las bandas que **no tienen ningún** concierto;
- los álbumes que cumplen algo respecto de **todas** sus canciones;
- el **primer** o **último** álbum de cada banda;
- entidades que deben compararse contra un conjunto de IDs o nombres calculado antes.

Tipos comunes de subconsulta:

- **Subconsulta escalar**: devuelve un solo valor.
- **Subconsulta de conjunto**: devuelve varios valores para usar con `IN` o `NOT IN`.
- **Subconsulta correlacionada**: depende de la fila externa y se evalúa en relación con ella.

Ejemplo de subconsulta escalar:

```sql
SELECT nombre
FROM albumes
WHERE anio_lanzamiento = (
    SELECT MIN(anio_lanzamiento)
    FROM albumes
);
```

La subconsulta calcula el año mínimo, y la consulta externa busca qué álbumes tienen ese valor.

### `IN`, `NOT IN`, `EXISTS` y `NOT EXISTS`

Estas herramientas aparecen mucho en la guía, pero no significan exactamente lo mismo.

- `IN`: se usa cuando quiero saber si un valor pertenece a un conjunto.
- `NOT IN`: se usa cuando quiero excluir los valores que aparecen en ese conjunto.
- `EXISTS`: no compara un valor puntual; solo pregunta si existe al menos una fila relacionada.
- `NOT EXISTS`: pregunta si no existe ninguna fila que cumpla la relación.

Ejemplo con `IN`:

```sql
SELECT nombre
FROM bandas
WHERE id IN (
    SELECT banda_id
    FROM albumes
    WHERE anio_lanzamiento <= 1980
);
```

Lógica:

- primero se obtienen los `banda_id` que tienen un álbum hasta 1980;
- después se muestran las bandas cuyos IDs están en ese conjunto.

Ejemplo con `NOT EXISTS`:

```sql
SELECT bandas.nombre
FROM bandas
WHERE NOT EXISTS (
    SELECT 1
    FROM conciertos_musicos
    WHERE conciertos_musicos.banda_id = bandas.id
);
```

Lógica:

- por cada banda, se revisa si existe alguna fila relacionada en `conciertos_musicos`;
- si no existe ninguna, esa banda no tiene conciertos registrados.

### Subconsultas correlacionadas

Una subconsulta correlacionada es una subconsulta que usa datos de la fila externa.

Eso significa que **no se resuelve una sola vez para toda la consulta**, sino una vez por cada fila que se está evaluando.

Ejemplo típico de la guía: obtener el primer álbum de cada banda.

```sql
SELECT bandas.nombre, albumes.nombre, albumes.anio_lanzamiento
FROM bandas
JOIN albumes
    ON albumes.banda_id = bandas.id
WHERE albumes.anio_lanzamiento = (
    SELECT MIN(albumes_2.anio_lanzamiento)
    FROM albumes AS albumes_2
    WHERE albumes_2.banda_id = bandas.id
);
```

Qué pasa acá:

- la consulta externa recorre bandas y álbumes;
- la subconsulta mira solo los álbumes de la banda actual;
- `MIN` encuentra el año del primer álbum de esa banda;
- el `WHERE` deja solo el álbum que coincide con ese mínimo.

Si la consigna pidiera el último álbum, la lógica sería la misma, cambiando `MIN` por `MAX`.

### La lógica de "todos"

Una de las ideas más difíciles de la guía es cuando la consigna dice que algo debe cumplir una condición para **todos** los casos relacionados.

Ejemplos:

- bandas cuyos álbumes **todos** duran más de 50 minutos;
- bandas que participaron en **todos** los conciertos de Argentina;
- álbumes cuyas canciones **todas** tienen ranking peor o igual a 30.

Muchas veces esto no se resuelve preguntando por lo que sí cumple, sino por lo que **rompe** la condición.

Idea típica:

- quiero bandas cuyos álbumes todos duren más de 50;
- entonces busco bandas para las que **no exista** ningún álbum con duración menor o igual a 50.

Ejemplo:

```sql
SELECT bandas.nombre
FROM bandas
WHERE NOT EXISTS (
    SELECT 1
    FROM albumes
    WHERE albumes.banda_id = bandas.id
      AND albumes.duracion <= 50
);
```

Esta lógica es muy importante:

- no pregunto por los álbumes buenos;
- pregunto si existe algún álbum que contradiga la regla;
- si no existe ninguno, entonces todos cumplen.

### Nivel 0

Acá predominan las consultas más básicas.

- Mostrar todos los campos de una tabla.
- Mostrar solo algunas columnas.
- Ordenar por fecha, nombre, ranking o duración.
- No hace falta relacionar tablas ni agrupar.

Idea general:

- identificar la tabla correcta;
- elegir columnas con `SELECT`;
- ordenar con `ORDER BY` si la consigna lo pide.

Ejemplo mental:

- si la consigna dice "nombre y fecha de todos los conciertos", ya sé que necesito solo `conciertos`;
- si además dice "ordenados por fecha", agrego `ORDER BY fecha`.

### Nivel 1

Aparecen filtros directos y recortes de resultados.

- Filtrar por igualdad: país, género, cantidad exacta de integrantes.
- Filtrar por comparación: mayor, menor, menor o igual.
- Combinar filtro con ordenamiento.
- Usar `LIMIT` para quedarse con una cantidad fija.

Idea general:

- primero se restringen las filas con `WHERE`;
- después se ordena con `ORDER BY`;
- al final se corta con `LIMIT` si hace falta.

Ojo con una idea importante:

- si la consigna pide "los mejores 7" y además "ordenados alfabéticamente", hay que distinguir entre
  seleccionar cuáles son esos 7
  y decidir cómo mostrarlos.
- a veces eso obliga a usar una subconsulta: primero se arma el top, después se reordena para mostrar.

### Nivel 2

Empiezan las relaciones entre tablas y los agregados simples.

- `JOIN` para traer álbumes de una banda o conciertos a los que asistió una banda.
- `DISTINCT` para no repetir bandas cuando una relación genera varias filas.
- `SUM` para acumular duraciones.
- `COUNT` para contar canciones o participaciones.

Idea general:

- si la consigna nombra dos entidades, probablemente hace falta `JOIN`;
- si pide totales, aparece una función agregada;
- si una banda puede salir varias veces, conviene pensar en `DISTINCT`.

Ejemplo:

```sql
SELECT DISTINCT bandas.nombre
FROM bandas
JOIN albumes
    ON albumes.banda_id = bandas.id
WHERE albumes.duracion < 40;
```

Sin `DISTINCT`, una banda saldría repetida si tuviera más de un álbum de menos de 40 minutos.

### Nivel 3

Acá aparecen subconsultas y condiciones más lógicas.

- `NOT EXISTS` para expresar que no debe existir ningún caso que rompa la condición.
- Subconsultas con `IN` para filtrar según otra consulta.
- `GROUP BY` + `COUNT` para obtener el concierto con más bandas o la banda con más canciones.
- Combinación de `JOIN`, filtros y ordenamientos más específicos.

Idea general:

- si la consigna dice "todos" o "ninguno", suele aparecer `NOT EXISTS`;
- si pide comparar contra un conjunto de IDs o nombres, sirve `IN`;
- si pide el máximo de algo, conviene contar y ordenar.

Ejemplo típico de este nivel:

- "bandas cuyos álbumes todos duran más de 50";
- no se busca un álbum bueno;
- se descarta la banda si aparece algún álbum malo.

Ese cambio de mirada es muy importante en SQL.

### Nivel 4

Es el nivel con más combinación de técnicas.

- Subconsultas correlacionadas con `MIN` o `MAX` para primer o último álbum.
- `AVG`, `MIN`, `COUNT` y `ROUND` para promedios y resúmenes por grupo.
- `HAVING` para filtrar grupos según su promedio o cantidad.
- `NOT IN` o `NOT EXISTS` para exclusiones.
- Doble `NOT EXISTS` para la idea de "estuvo en todos los conciertos de Argentina".
- Comparaciones entre entidades relacionadas, como ranking de canción vs ranking de álbum.

Idea general:

- identificar si la consigna pide una respuesta por fila o por grupo;
- si pide resumen por banda o género, usar `GROUP BY`;
- si pide ausencia total, pensar en `NOT IN` o `NOT EXISTS`;
- si pide que todas las relacionadas cumplan algo, usar lógica universal.

Ejemplo importante:

```sql
SELECT bandas.nombre
FROM bandas
WHERE NOT EXISTS (
    SELECT 1
    FROM conciertos
    WHERE conciertos.pais = 'Argentina'
      AND NOT EXISTS (
          SELECT 1
          FROM conciertos_musicos
          WHERE conciertos_musicos.banda_id = bandas.id
            AND conciertos_musicos.concierto_id = conciertos.id
      )
);
```

Lógica:

- recorro cada banda;
- busco si existe algún concierto argentino al que no haya ido;
- si no existe ninguno, entonces estuvo en todos.

### Patrones que más se repiten

- `SELECT ... FROM ... WHERE ...`: base para mostrar y filtrar.
- `SELECT ... FROM ... JOIN ... WHERE ...`: cuando la consigna mezcla varias tablas.
- `SELECT DISTINCT ...`: cuando una misma entidad puede salir repetida.
- `SELECT ... GROUP BY ... HAVING ...`: cuando se resumen datos por banda, género o concierto.
- `WHERE ... IN (subconsulta)`: cuando una tabla depende del resultado de otra.
- `WHERE ... NOT IN (subconsulta)`: cuando se quieren excluir casos.
- `WHERE NOT EXISTS (subconsulta)`: cuando se expresa que no debe existir ninguna excepción.
- `ORDER BY ... LIMIT ...`: cuando importa top, peor, primero o último.

Lo importante no es memorizar la forma exacta, sino reconocer el patrón de fondo.

### Cómo elegir la lógica correcta

- Si la consigna pide columnas de una sola tabla: `SELECT` simple.
- Si pide cruzar información de varias tablas: `JOIN`.
- Si pide evitar repetidos: `DISTINCT`.
- Si pide sumar, promediar, contar, mínimo o máximo: agregadas.
- Si pide una condición sobre grupos: `GROUP BY` + `HAVING`.
- Si dice "al menos uno": `EXISTS`, `IN` o un `JOIN` bien filtrado.
- Si dice "ninguno": `NOT EXISTS` o `NOT IN`.
- Si dice "todos": lógica universal, muchas veces con doble `NOT EXISTS`.
- Si pide primero, último, mejor o peor: ordenar, usar `LIMIT`, o comparar con `MIN` / `MAX`.

Pregunta guía útil antes de escribir SQL:

- ¿la respuesta sale de una tabla o de varias?
- ¿quiero filas individuales o un resumen por grupo?
- ¿la condición es directa o depende de otra consulta?
- ¿la consigna habla de alguno, ninguno o todos?

Si respondés eso, normalmente ya queda bastante claro qué herramienta usar.